# Feature extraction: parcellated BOLD -> block-mean features

This notebook produces the input features for the decoding analyses. For each subject and run, it:

1. **Parcellates** the fMRIPrep-preprocessed BOLD into mean signal per atlas region
2. **Denoises** by regressing out confound columns (motion, aCompCor, cosine high-pass)
3. **Computes a spike mask** based on framewise displacement
4. **Saves** per-run `.npy` files for parcel timeseries + spike mask + provenance JSON

Then it builds **block-level features** (one row per Thoughtblock or SART block) by HRF-shifting block TRs and computing the mean across non-spike TRs.

## Decisions you can change in `config.py`

- **Atlas**: `ATLAS_NAME` and `ATLAS_KWARGS`. Default: Yeo 17-network. Swap to Schaefer or any nilearn atlas.
- **Denoising**: `ACTIVE_CONFOUND_GROUPS` (list of group keys from `CONFOUND_GROUPS`). Default: motion-24 + aCompCor-5 + cosine-12 high-pass.
- **Block QC**: `MIN_FRACTION_USABLE` and `FD_SPIKE_THRESHOLD` (defaults 0.7, 0.6mm).
- **HRF lag**: `HRF_LAG_TRS` (default 3 TRs = 6s).

All decisions are written to `provenance.json` per run, so you can inspect what was applied.

## Outputs

- `derivatives/parcels/<atlas_name>/sub-XXX_run-NN_parcel_ts.npy` — (n_TRs, n_parcels)
- `derivatives/parcels/<atlas_name>/sub-XXX_run-NN_spike_mask.npy` — (n_TRs,) bool
- `derivatives/parcels/<atlas_name>/sub-XXX_run-NN_provenance.json` — metadata
- `derivatives/block_features.csv` — one row per (subject, run, ThoughtNumber, condition), with parcel-mean columns + dyscontrol/probe metadata

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))

from config import (
    BASE, FMRIPREP_ROOT, DATA_ROOT, DERIVATIVES_DIR, PARCELS_DIR,
    TRIAL_LEVEL_CSV, BLOCK_FEATURES_CSV,
    EXCLUDE_SUBJECTS, RUNS, TASK, SESSION, TR_SECONDS, HRF_LAG_TRS,
    ATLAS_NAME, ATLAS_KWARGS, YEO17_NAMES, SMOOTHING_FWHM,
    CONFOUND_GROUPS, ACTIVE_CONFOUND_GROUPS,
    FD_SPIKE_THRESHOLD, MIN_FRACTION_USABLE,ATLAS_SUBDIR,
)
from parcellation import (
    load_atlas, get_bold_paths, build_confound_matrix,
    process_run, extract_block_features,
)

from events_loader import load_events_tsv, block_event_rows

DERIVATIVES_DIR.mkdir(parents=True, exist_ok=True)
ATLAS_OUT_DIR = PARCELS_DIR / ATLAS_SUBDIR
ATLAS_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Atlas: {ATLAS_NAME} (kwargs={ATLAS_KWARGS})')
print(f'Active denoising: {ACTIVE_CONFOUND_GROUPS}')
print(f'FD spike threshold: {FD_SPIKE_THRESHOLD}')
print(f'Smoothing: {SMOOTHING_FWHM}'),
print(f'ATLAS_NAME    = {ATLAS_NAME}')
print(f'ATLAS_SUBDIR  = {ATLAS_SUBDIR}')
print(f'ATLAS_OUT_DIR = {ATLAS_OUT_DIR}')
print(f'SMOOTHING_FWHM = {SMOOTHING_FWHM}')

## Step 1: Load atlas

In [ ]:
atlas_img, atlas_labels = load_atlas(ATLAS_NAME, ATLAS_KWARGS)

# Override labels for Yeo-17 with our standard names
if ATLAS_NAME == 'yeo_2011' and ATLAS_KWARGS.get('n_networks') == 17:
    atlas_labels = list(YEO17_NAMES)

print(f'Atlas image: shape={atlas_img.shape}, dtype={atlas_img.get_fdata().dtype}')
print(f'Number of parcels: {len(atlas_labels)}')
print(f'First 5 labels: {atlas_labels[:5]}')
print(f'Unique non-zero values: {sorted(np.unique(atlas_img.get_fdata()))[1:6]}...')

### Spot-check: visualize the atlas

Rough sanity check that the atlas loaded correctly.

In [ ]:
from nilearn import plotting
fig = plotting.plot_roi(
    atlas_img, title=f'{ATLAS_NAME} parcels (axial slices)',
    cut_coords=list(range(-30, 60, 15)), display_mode='z',
)
plt.show()

## Step 2: Run parcellation for one subject (test)

Test the pipeline end-to-end on sub-302 run-01 before scaling up.

In [ ]:
test_sub = 329
test_run = '03'

result = process_run(
    subject=test_sub, run=test_run,
    fmriprep_root=FMRIPREP_ROOT,
    atlas_img=atlas_img,
    output_dir=ATLAS_OUT_DIR,
    session=SESSION, task=TASK, tr=TR_SECONDS,
    confound_groups=CONFOUND_GROUPS,
    active_confound_groups=ACTIVE_CONFOUND_GROUPS,
    fd_spike_threshold=FD_SPIKE_THRESHOLD,
    smoothing_fwhm=SMOOTHING_FWHM,
    overwrite=True,                   # <-- force redo of test
)
print(f'\nResult: {result}')

# Read provenance to check
with open(result['provenance_path']) as f:
    prov = json.load(f)
print(f'\nProvenance:')
for k, v in prov.items():
    if k == 'confound_columns_used':
        print(f'  {k}: {v[:5]}... (n={len(v)})')
    else:
        print(f'  {k}: {v}')

### Spot-check: visualize the parcellated timeseries

Plot all parcels' time courses for the test run. Should look like reasonable BOLD signal -- positive and negative excursions, no obvious artifacts.

In [ ]:
parcel_ts = np.load(result['parcel_ts_path'])
spike_mask = np.load(result['spike_mask_path'])

# Z-score each parcel for visualization (centered + unit-variance per parcel)
parcel_z = (parcel_ts - parcel_ts.mean(axis=0)) / parcel_ts.std(axis=0)

print(f'parcel_ts shape: {parcel_ts.shape}')
print(f'spike TRs: {spike_mask.sum()} / {len(spike_mask)}')

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

ax = axes[0]
for i in range(parcel_z.shape[1]):
    ax.plot(parcel_z[:, i], alpha=0.5, lw=0.7,
            label=atlas_labels[i] if i < 5 else None)
ax.set_ylabel('Parcel signal (z-scored for display)')
ax.set_title(f'sub-{test_sub} run-{test_run}: parcellated timeseries')
ax.legend(loc='upper right', fontsize=7)

ax = axes[1]
ax.plot(spike_mask.astype(int), 'r-', drawstyle='steps-post')
ax.set_ylabel('Spike TR (1=excluded)')
ax.set_xlabel('TR')
ax.set_ylim(-0.1, 1.1)
ax.set_title(f'Spike mask (FD > {FD_SPIKE_THRESHOLD}mm)')

plt.tight_layout()
plt.show()

## Step 3: Run parcellation for all subjects/runs

This is the slow step. Takes a few minutes per run. Skips runs with already-existing outputs unless `overwrite=True`.

In [ ]:
# Get list of subjects from trial_level_merged
trial_level = pd.read_csv(TRIAL_LEVEL_CSV)
subjects = sorted(trial_level['subject'].unique())
print(f'{len(subjects)} subjects to process')

results_summary = []
for sub in subjects:
    if sub in EXCLUDE_SUBJECTS:
        continue
    for run in RUNS:
        result = process_run(
            subject=sub, run=run,
            fmriprep_root=FMRIPREP_ROOT,
            atlas_img=atlas_img,
            output_dir=ATLAS_OUT_DIR,
            session=SESSION, task=TASK, tr=TR_SECONDS,
            confound_groups=CONFOUND_GROUPS,
            active_confound_groups=ACTIVE_CONFOUND_GROUPS,
            fd_spike_threshold=FD_SPIKE_THRESHOLD,
            smoothing_fwhm=SMOOTHING_FWHM,   # <-- this line
            overwrite=True,                   # <-- force redo of test
        )
        if result is not None:
            results_summary.append({
                'subject': sub,
                'run': run,
                **{k: v for k, v in result.items() if not k.endswith('_path')}
            })

summary_df = pd.DataFrame(results_summary)
print(f'\nProcessed {len(summary_df)} runs across {summary_df["subject"].nunique()} subjects')
print(f'\nSpike TRs per run distribution:')
print(summary_df['n_spike_trs'].describe().round(1))

## Step 4: Build block-level feature DataFrame

Iterate over all blocks (Thoughtblock and SART) and extract HRF-shifted block-mean parcel features. One row per block.

In [ ]:
def qc_block_extraction(subject, run, max_blocks_to_show=None):
    """
    Side-by-side comparison: raw events.tsv rows vs extracted block summary.
    Prints both Thoughtblocks and SART blocks with onset/duration.
    """
    ev = load_events_tsv(subject, run)
    if ev is None:
        print(f'No events.tsv for sub-{subject} run-{run}')
        return

    print(f'=== sub-{subject} run-{run} ===')
    print(f'events.tsv: {len(ev)} rows, columns={list(ev.columns)}')
    print(f'Trial types: {ev["trial_type"].value_counts().to_dict()}')
    if 'eventType' in ev.columns:
        print(f'Event types: {ev["eventType"].value_counts().to_dict()}')
    print()

    blocks = block_event_rows(ev)
    print(f'Extracted {len(blocks)} blocks total: '
          f'{sum(b["condition"]=="Thoughtblock" for b in blocks)} TB + '
          f'{sum(b["condition"]=="SART" for b in blocks)} SART')
    print()

    # ---- Thoughtblocks ----
    print('THOUGHTBLOCKS')
    print('-' * 90)
    print(f'{"thoughtNum":>10s}  {"onset":>8s}  {"duration":>8s}  thought')
    print('-' * 90)
    if 'eventType' in ev.columns:
        tb_raw = ev[ev['eventType'] == 'Thoughtblock'].sort_values('onset')
        # Show the raw rows
        print('  RAW from events.tsv:')
        for _, r in tb_raw.iterrows():
            tn = int(r['thoughtNumber'])
            thought = r.get('Thought', '')
            print(f'    {tn:>8d}  {r["onset"]:>8.2f}  {r["duration"]:>8.2f}  {thought}')
        # Show extracted
        print('\n  EXTRACTED from block_event_rows():')
        tb_blocks = [b for b in blocks if b['condition'] == 'Thoughtblock']
        for b in sorted(tb_blocks, key=lambda x: x['onset']):
            print(f'    {b["thoughtNumber"]:>8d}  {b["onset"]:>8.2f}  '
                  f'{b["duration"]:>8.2f}  {b["thought"]}')
    print()

    # ---- SART blocks ----
    print('SART BLOCKS (aggregated from go/nogo trials by thoughtNumber)')
    print('-' * 90)
    sart_raw = ev[ev['trial_type'].isin(['go', 'nogo'])]
    if 'thoughtNumber' in sart_raw.columns:
        print(f'{"thoughtNum":>10s}  {"n_trials":>8s}  {"onset":>8s}  '
              f'{"end":>8s}  {"duration":>8s}  thought')
        print('-' * 90)
        # Show raw aggregation
        print('  RAW (computed from go/nogo trials):')
        for tn, g in sart_raw.groupby('thoughtNumber'):
            onset = g['onset'].min()
            end = (g['onset'] + g['duration']).max()
            duration = end - onset
            thought = g['Thought'].iloc[0] if 'Thought' in g.columns else ''
            print(f'    {int(tn):>8d}  {len(g):>8d}  {onset:>8.2f}  '
                  f'{end:>8.2f}  {duration:>8.2f}  {thought}')
        # Show extracted
        print('\n  EXTRACTED from block_event_rows():')
        sart_blocks = [b for b in blocks if b['condition'] == 'SART']
        for b in sorted(sart_blocks, key=lambda x: x['thoughtNumber']):
            print(f'    {b["thoughtNumber"]:>8d}  {"--":>8s}  '
                  f'{b["onset"]:>8.2f}  {b["onset"]+b["duration"]:>8.2f}  '
                  f'{b["duration"]:>8.2f}  {b["thought"]}')
    print()


# ---- Run on test subject ----
qc_block_extraction(351, '02')

In [ ]:
# Build the full block-feature DataFrame
print('Building block features for all subjects...')

all_block_rows = []
n_dropped_qc = 0
n_no_parcels = 0
n_no_events = 0

for sub in subjects:
    if sub in EXCLUDE_SUBJECTS:
        continue
    for run in RUNS:
        # Load parcels and spike mask
        parcel_ts_path = ATLAS_OUT_DIR / f'sub-{sub}_run-{run}_parcel_ts.npy'
        spike_mask_path = ATLAS_OUT_DIR / f'sub-{sub}_run-{run}_spike_mask.npy'
        if not parcel_ts_path.exists():
            n_no_parcels += 1
            continue
        parcel_ts = np.load(parcel_ts_path)
        spike_mask = np.load(spike_mask_path)
        
        # Load events
        events = load_events_tsv(sub, run)
        if events is None:
            n_no_events += 1
            continue
        blocks = block_event_rows(events)
        
        # Extract block features
        for block in blocks:
            block_mean, qc = extract_block_features(
                parcel_ts, spike_mask,
                onset_sec=block['onset'],
                duration_sec=block['duration'],
                tr=TR_SECONDS,
                hrf_lag_trs=HRF_LAG_TRS,
                min_fraction_usable=MIN_FRACTION_USABLE,
            )
            if block_mean is None:
                n_dropped_qc += 1
                continue
            row = {
                'subject': sub,
                'run': int(run),
                'condition': block['condition'],
                'thoughtNumber': block['thoughtNumber'],
                'onset': block['onset'],
                'duration': block['duration'],
                **qc,
            }
            for i, val in enumerate(block_mean):
                row[f'parcel_{i:02d}'] = val
            all_block_rows.append(row)

block_df = pd.DataFrame(all_block_rows)
print(f'\nBlock-feature DataFrame: {block_df.shape}')
print(f'  Dropped (QC fail, fraction_usable < {MIN_FRACTION_USABLE}): {n_dropped_qc}')
print(f'  Skipped (no parcels file): {n_no_parcels}')
print(f'  Skipped (no events file):  {n_no_events}')
print()
print(f'Blocks per condition:')
print(block_df['condition'].value_counts())
print()
print(f'Blocks per subject:')
print(block_df.groupby('subject').size().describe().round(1))
print(block_df.head(10))         # first 10 rows

## Step 5: Merge with trial-level data

Join in `dyscontrol_imputed`, probes, and trait scores so block_features has all the context for downstream decoding.

In [ ]:
# Merge with trial_level on (subject, run, ThoughtNumber)
# Note: trial_level uses 'ThoughtNumber' (capitalized); block_df uses 'thoughtNumber'
block_with_meta = block_df.merge(
    trial_level.rename(columns={'ThoughtNumber': 'thoughtNumber'}),
    on=['subject', 'run', 'thoughtNumber'],
    how='left',
    suffixes=('', '_trial'),
)

print(f'Merged shape: {block_with_meta.shape}')
print(f'Rows with dyscontrol_imputed: {block_with_meta["dyscontrol_imputed"].notna().sum()}')
print(f'Rows missing dyscontrol_imputed:')
missing = block_with_meta[block_with_meta['dyscontrol_imputed'].isna()]
if len(missing) > 0:
    print(missing[['subject', 'run', 'thoughtNumber', 'condition']].head(10))

# Save
block_with_meta.to_csv(BLOCK_FEATURES_CSV, index=False)
print(f'\nSaved: {BLOCK_FEATURES_CSV}')

In [ ]:
# Spot-check block_with_meta against ALL source files
# - alldata: probes
# - prescan: dyscontrol_imputed
# - traits: PTQ/PSWQ
# - parcellation .npy: that the parcel values in the row really are
#   the HRF-shifted block-mean from the right TRs

np.random.seed(42)
n_check = 5

# Re-load source data (these were defined in data_merge.ipynb)
from alldata_loader import load_alldata
from questionnaire_loader import load_prescan_thought_ratings, load_trait_scores
from merge import load_manual_corrections, apply_manual_corrections
from config import (
    ALLDATA_PATH, PRESCAN_PATH, QUALTRICS_PATH, MANUAL_CORRECTIONS_PATH,
)

alldata = load_alldata(ALLDATA_PATH)
prescan = load_prescan_thought_ratings(PRESCAN_PATH)
traits = load_trait_scores(QUALTRICS_PATH)

corrections = load_manual_corrections(MANUAL_CORRECTIONS_PATH)
alldata_corrected = apply_manual_corrections(alldata, corrections)

print(f'alldata_corrected: {alldata_corrected.shape}')
print(f'prescan: {prescan.shape}')
print(f'traits: {traits.shape}')

# Sample 5 random rows
sample = block_with_meta.sample(n_check, random_state=42).reset_index(drop=True)

for _, row in sample.iterrows():
    sub = int(row['subject'])
    run = int(row['run'])
    tn = int(row['thoughtNumber'])
    condition = row['condition']
    thought = row.get('thought_key', '?')
    print(f'\n=== sub-{sub} run-{run} {condition} thoughtNumber={tn} '
          f'thought={thought!r} ===')

    # 1. ALLDATA: probes
    ad_match = alldata_corrected[
        (alldata_corrected['subject'] == sub)
        & (alldata_corrected['run'] == run)
        & (alldata_corrected['ThoughtNumber'] == tn)
    ]
    print(f'\n  ALLDATA: {len(ad_match)} matching rows')
    if len(ad_match) > 0:
        ad_row = ad_match.iloc[0]
        print(f'    PROBES (alldata vs merged):')
        for p in [f'probe{i}' for i in range(1, 7)]:
            if p in ad_row.index and p in row.index:
                ad_val = ad_row[p]
                m_val = row[p]
                match = '✓' if (pd.isna(ad_val) and pd.isna(m_val)) or ad_val == m_val else '✗'
                print(f'      {p}: alldata={ad_val}  merged={m_val}  {match}')

    # 2. PRESCAN: dyscontrol_imputed
    ps_match = prescan[
        (prescan['subject'] == sub)
        & (prescan['thought_key'] == thought)
    ]
    print(f'\n  PRESCAN: {len(ps_match)} matching rows')
    if len(ps_match) > 0:
        ps_val = ps_match['dyscontrol_imputed'].iloc[0]
        m_val = row['dyscontrol_imputed']
        match = '✓' if abs(ps_val - m_val) < 1e-6 else '✗'
        print(f'    dyscontrol_imputed: prescan={ps_val:.3f}  '
              f'merged={m_val:.3f}  {match}')

    # 3. TRAITS: PTQ/PSWQ
    tr_match = traits[traits['subject'] == sub]
    print(f'\n  TRAITS: {len(tr_match)} matching rows')
    if len(tr_match) > 0:
        for col in ['ptq_total', 'pswq_total']:
            tr_val = tr_match[col].iloc[0]
            m_val = row.get(col, np.nan)
            if pd.isna(tr_val) and pd.isna(m_val):
                match = '✓ (both NaN)'
            elif pd.isna(tr_val) or pd.isna(m_val):
                match = '✗ (one is NaN)'
            else:
                match = '✓' if abs(tr_val - m_val) < 1e-6 else '✗'
            print(f'    {col}: traits={tr_val}  merged={m_val}  {match}')

    # 4. PARCELLATION: re-extract block-mean from .npy and compare
    parcel_path = (PARCELS_DIR / ATLAS_SUBDIR
                    / f'sub-{sub}_run-{run:02d}_parcel_ts.npy')
    spike_path = (PARCELS_DIR / ATLAS_SUBDIR
                   / f'sub-{sub}_run-{run:02d}_spike_mask.npy')
    if not parcel_path.exists():
        print(f'\n  PARCELLATION: file not found at {parcel_path}')
    else:
        parcel_ts = np.load(parcel_path)
        spike_mask = np.load(spike_path)
        # Recompute block-mean using same logic as extract_block_features
        block_mean, qc = extract_block_features(
            parcel_ts, spike_mask,
            onset_sec=row['onset'],
            duration_sec=row['duration'],
            tr=TR_SECONDS,
            hrf_lag_trs=HRF_LAG_TRS,
            min_fraction_usable=MIN_FRACTION_USABLE,
        )
        print(f'\n  PARCELLATION: re-extracted from .npy')
        print(f'    QC: n_block_trs={qc["n_block_trs"]} (merged={row["n_block_trs"]}), '
              f'n_usable={qc["n_usable"]} (merged={row["n_usable"]})')
        # Compare 3 random parcel values
        print(f'    PARCEL VALUES (re-extracted vs merged):')
        for parcel_idx in [0, 8, 16]:
            re_val = block_mean[parcel_idx]
            m_val = row[f'parcel_{parcel_idx:02d}']
            match = '✓' if abs(re_val - m_val) < 1e-3 else '✗'
            print(f'      parcel_{parcel_idx:02d}: re-extracted={re_val:.4f}  '
                  f'merged={m_val:.4f}  {match}')

In [ ]:
df = pd.read_csv(BLOCK_FEATURES_CSV)
sub301_run1_tn1 = df[(df['subject']==301) & (df['run']==1) & (df['thoughtNumber']==1)]
print(sub301_run1_tn1[['condition', 'NoGo_ACC', 'AccIndex', 'Mean_go_RT', 'probe1']])

In [ ]:
# Inspect the 93 Thought.x vs Thought.y mismatches in raw alldata
import pandas as pd
import re

# Re-load raw alldata WITHOUT the .x/.y drop step in the loader
from pathlib import Path
from config import ALLDATA_PATH

with open(ALLDATA_PATH) as f:
    raw = f.read()

# Same line-fixing as in load_alldata
raw_lines = raw.split('\n')
fixed_lines = []
for line in raw_lines:
    if not line.strip():
        continue
    if re.match(r'^\d+,', line):
        fixed_lines.append(line)
    elif fixed_lines:
        fixed_lines[-1] = fixed_lines[-1] + ' ' + line.lstrip()
    else:
        fixed_lines.append(line)

# Easier path: read with pandas and skip the malformed rows for this diagnostic
ad_raw = pd.read_csv(ALLDATA_PATH, on_bad_lines='skip')
print(f'Loaded {len(ad_raw)} rows (some may have been skipped if malformed)')

# Normalize both
def norm(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    s = re.sub(r"[,.\-_/;:!?\\\"'()&\[\]{}]", ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

ad_raw['tx_norm'] = ad_raw['Thought.x'].apply(norm)
ad_raw['ty_norm'] = ad_raw['Thought.y'].apply(norm)

# Find mismatches where both are present
mismatch = ad_raw[
    ad_raw['tx_norm'].notna() & ad_raw['ty_norm'].notna()
    & (ad_raw['tx_norm'] != ad_raw['ty_norm'])
]

print(f'\n{len(mismatch)} rows where Thought.x != Thought.y after normalization')
print(f'\nUnique (Thought.x, Thought.y) pairs:')
unique_pairs = (
    mismatch[['participant', 'Thought.x', 'Thought.y']]
    .drop_duplicates(subset=['Thought.x', 'Thought.y'])
    .sort_values('participant')
)
print(unique_pairs.to_string(index=False))

In [ ]:
import json
with open(MANUAL_CORRECTIONS_PATH) as f:
    data = json.load(f)
corrections = data['corrections']
print(f'Loaded {len(corrections)} corrections')

# Build set of (subject, from_key) currently corrected
corrected = {(c['subject'], c['from']) for c in corrections}

# Check which mismatched typos are NOT in corrections
typo_candidates = [
    (307, 'overwheling'),                     # prescan typo, alldata clean
    (336, 'woodworking record cabinent'),     # alldata typo
    (346, 'religious oerformance'),           # alldata typo  
    (359, 'calorie count'),                   # alldata, singular
    (363, 'not fun enough'),                  # ambiguous (real distinct phrases)
    (374, 'job insecruity'),                  # alldata typo
    (376, 'bad minton'),                      # alldata typo
    (342, 'weather'),                         # ambiguous (clean alldata, weater in prescan)
]

print('\nChecking typo candidates against manual_corrections.json:')
for sub, key in typo_candidates:
    in_corrections = (sub, key) in corrected
    status = '✓ already corrected' if in_corrections else '✗ NOT in corrections'
    print(f'  sub-{sub:3d} "{key:35s}": {status}')

### Spot-check: probe-dyscontrol within-subject correlations

Same sanity check as in `data_merge.ipynb` -- this validates that the block_features table has its trial labels correctly aligned (same merge logic, but operating on the post-parcellation rows that survived QC).

In [ ]:
from scipy.stats import pearsonr

tb = block_with_meta[block_with_meta['condition'] == 'Thoughtblock']

print('Within-subject correlations between dyscontrol_imputed and probes (Thoughtblock):')
print(f"{'probe':6s}  {'mean r':>8s}  {'%pos':>5s}  {'n_subj':>6s}")
print('-' * 40)
for p_num in range(1, 7):
    col = f'probe{p_num}'
    if col not in tb.columns:
        continue
    rs = []
    for sub, g in tb.groupby('subject'):
        valid = g.dropna(subset=['dyscontrol_imputed', col])
        if len(valid) < 5 or valid['dyscontrol_imputed'].std() == 0 or valid[col].std() == 0:
            continue
        r, _ = pearsonr(valid['dyscontrol_imputed'], pd.to_numeric(valid[col], errors='coerce'))
        if not np.isnan(r):
            rs.append(r)
    pct_pos = 100 * np.mean(np.array(rs) > 0)
    print(f'{col:6s}  {np.mean(rs):>+8.3f}  {pct_pos:>4.0f}%  {len(rs):>6d}')
print()
print('(Expected from earlier work: probe1 r ≈ 0.46, probe4 ≈ 0.38, probe5 ≈ 0.44, ~80%+ positive)')

# For QC purposes: aggregating spike counts and percentages

## Original analyses used spikes generated by fmriprep, comprising DVARS and FD rules. Exclusions were decided per entire run.
## For these analyses, we used FD only (output by niftiLabelsMasker), and excluded by block (as this is the unit that the model trains on).

In [ ]:
# =============================================================================
# Aggregate spike statistics across subjects/runs from saved spike masks.
# Produces per-subject and group-level summaries; saves to CSV.
# =============================================================================
import numpy as np
import pandas as pd
from pathlib import Path

REST_TASK = 'restingstate'
REST_ATLAS_SUBDIR = f'{ATLAS_NAME}_rest'
REST_PARCELS_DIR = PARCELS_DIR / REST_ATLAS_SUBDIR
ATLAS_DIR = ATLAS_OUT_DIR

rows = []

# --- Rest scans ---
for spike_path in sorted(REST_PARCELS_DIR.glob('sub-*_run-01_spike_mask.npy')):
    sub = int(spike_path.stem.split('sub-')[1].split('_')[0])
    mask = np.load(spike_path)
    rows.append({
        'subject':  sub,
        'scan':     'rest',
        'run':      1,
        'n_trs':    len(mask),
        'n_spike':  int(mask.sum()),
        'pct_spike': 100 * mask.mean(),
    })

# --- Task runs (SART + Thoughtblock) ---
for spike_path in sorted(ATLAS_DIR.glob('sub-*_run-*_spike_mask.npy')):
    stem = spike_path.stem
    sub = int(stem.split('sub-')[1].split('_')[0])
    run = int(stem.split('run-')[1].split('_')[0])
    mask = np.load(spike_path)
    rows.append({
        'subject':  sub,
        'scan':     'task',
        'run':      run,
        'n_trs':    len(mask),
        'n_spike':  int(mask.sum()),
        'pct_spike': 100 * mask.mean(),
    })

spike_df = pd.DataFrame(rows)
print(f'Aggregated spike stats across {len(spike_df)} scan-runs '
      f'({spike_df["subject"].nunique()} subjects)')

# Per-subject summary (long)
per_sub = (spike_df.groupby(['subject', 'scan'])
                    .agg(n_trs=('n_trs', 'sum'),
                         n_spike=('n_spike', 'sum'))
                    .assign(pct_spike=lambda d: 100 * d['n_spike'] / d['n_trs'])
                    .reset_index())

# Group-level: mean and range per scan type
print('\n=== Group-level spike stats by scan type ===')
for scan, g in per_sub.groupby('scan'):
    print(f'\n{scan.upper()} scans (n={len(g)} subjects):')
    print(f'  % spike TRs: M = {g["pct_spike"].mean():.2f}, '
          f'SD = {g["pct_spike"].std():.2f}, '
          f'range = {g["pct_spike"].min():.2f}-{g["pct_spike"].max():.2f}')
    print(f'  Total TRs: {g["n_trs"].sum():,}')
    print(f'  Total spike TRs: {g["n_spike"].sum():,}')

# --- Wide format: one column for rest, one for task ---
per_sub_wide = per_sub.pivot(index='subject', columns='scan',
                             values=['n_trs', 'n_spike', 'pct_spike'])
# Flatten MultiIndex columns -> e.g. pct_spike_rest, pct_spike_task
per_sub_wide.columns = [f'{stat}_{scan}' for stat, scan in per_sub_wide.columns]
per_sub_wide = per_sub_wide.reset_index()

spike_df.to_csv('figure_data/spike_summary_per_run.csv', index=False)
per_sub_wide.to_csv('figure_data/spike_summary_per_subject.csv', index=False)
print(f'\nSaved to figure_data/spike_summary_*.csv')

In [ ]:
# collates DVARS and FD motion data for all participants and participants with big differences.

import numpy as np
import pandas as pd
from pathlib import Path

CONFOUNDS_ROOT = Path("/ix1/lhallion/TReW-Brain/preprocessed/")   # fmriprep root (contains sub-*/ses-01/func)
SPIKE_MASK_DIR = Path("/ihome/cladouceur/westbroo/Scripts/lhallion_scripts/TReW-Brain/dyscontrol_mvpa_featlike_nosmoothing/derivatives/parcels/yeo_2011_motiononly/")   # same dir process_run() wrote spike_mask.npy into

rows = []
for spike_path in sorted(SPIKE_MASK_DIR.glob('sub-*_run-*_spike_mask.npy')):
    stem = spike_path.stem
    sub = int(stem.split('sub-')[1].split('_')[0])
    run = stem.split('run-')[1].split('_')[0]

    confounds_path = (CONFOUNDS_ROOT / f'sub-{sub}' / 'ses-01' / 'func'
                       / f'sub-{sub}_ses-01_task-PDT_run-{run}_desc-confounds_timeseries.tsv')
    if not confounds_path.exists():
        print(f'missing confounds: sub-{sub} run-{run}')
        continue

    df = pd.read_csv(confounds_path, sep='\t')
    spike_mask = np.load(spike_path)

    outlier_cols = [c for c in df.columns if c.startswith('motion_outlier')]
    outlier_trs = set(np.where(df[outlier_cols].sum(axis=1) > 0)[0]) if outlier_cols else set()
    spike_trs = set(np.where(spike_mask)[0])
    n_trs = len(df)

    rows.append({
        'subject': sub, 'run': run, 'n_trs': n_trs,
        'n_motion_outlier': len(outlier_trs),
        'n_spike_mask': len(spike_trs),
        'n_missed_by_spike_mask': len(outlier_trs - spike_trs),
        'pct_outlier': 100 * len(outlier_trs) / n_trs,
        'pct_spike_mask': 100 * len(spike_trs) / n_trs,
    })

comp_df = pd.DataFrame(rows)

per_sub = (comp_df.groupby('subject')
           .agg(n_trs=('n_trs', 'sum'),
                n_motion_outlier=('n_motion_outlier', 'sum'),
                n_spike_mask=('n_spike_mask', 'sum'))
           .assign(pct_outlier=lambda d: 100 * d['n_motion_outlier'] / d['n_trs'],
                   pct_spike_mask=lambda d: 100 * d['n_spike_mask'] / d['n_trs'],
                   gap_pts=lambda d: d['pct_outlier'] - d['pct_spike_mask'])
           .sort_values('pct_outlier', ascending=False))

print(per_sub.head(20).to_string())
comp_df.to_csv('figure_data/spike_comparison_per_run.csv', index=False)
per_sub.to_csv('figure_data/spike_comparison_per_subject.csv', index=False)

In [ ]:
# view the full dataset
print(per_sub)

In [ ]:
# prints the number of blocks eliminated per subject based on FD spike percentages

import numpy as np
import pandas as pd
from pathlib import Path
from parcellation import get_block_indices, extract_block_features
from events_loader import load_events_tsv, block_event_rows

CONFOUNDS_ROOT = Path("/ix1/lhallion/TReW-Brain/preprocessed/")   # fmriprep root (contains sub-*/ses-01/func)
SPIKE_MASK_DIR = Path("/ihome/cladouceur/westbroo/Scripts/lhallion_scripts/TReW-Brain/dyscontrol_mvpa_featlike_nosmoothing/derivatives/parcels/yeo_2011_motiononly/")   # same dir process_run() wrote spike_mask.npy into

TR = 2.0
HRF_LAG_TRS = 3
MIN_FRACTION_USABLE = 0.7
EXCLUDE_THRESHOLD = 18  # 2+ runs worth of dropped blocks — same bar used for sub-351

def build_corrected_spike_mask(confounds_path):
    df = pd.read_csv(confounds_path, sep='\t')
    outlier_cols = [c for c in df.columns if c.startswith('motion_outlier')]
    if outlier_cols:
        return (df[outlier_cols].sum(axis=1) > 0).values
    return np.zeros(len(df), dtype=bool)

rows = []
for spike_path in sorted(SPIKE_MASK_DIR.glob('sub-*_run-*_spike_mask.npy')):
    stem = spike_path.stem
    sub = int(stem.split('sub-')[1].split('_')[0])
    run = stem.split('run-')[1].split('_')[0]

    confounds_path = (CONFOUNDS_ROOT / f'sub-{sub}' / 'ses-01' / 'func'
                       / f'sub-{sub}_ses-01_task-PDT_run-{run}_desc-confounds_timeseries.tsv')
    if not confounds_path.exists():
        continue

    corrected_mask = build_corrected_spike_mask(confounds_path)
    n_trs = len(corrected_mask)

    events_df = load_events_tsv(sub, run)
    tb_blocks = [b for b in block_event_rows(events_df) if b['condition'] == 'Thoughtblock']

    for block in tb_blocks:
        _, qc = extract_block_features(
            parcel_ts=np.zeros((n_trs, 1)),
            spike_mask=corrected_mask,
            onset_sec=block['onset'],
            duration_sec=block['duration'],
            tr=TR,
            hrf_lag_trs=HRF_LAG_TRS,
            min_fraction_usable=MIN_FRACTION_USABLE,
        )
        rows.append({'subject': sub, 'run': run,
                      'thoughtNumber': block['thoughtNumber'],
                      'fraction_usable': qc['fraction_usable'],
                      'dropped': qc['fraction_usable'] < MIN_FRACTION_USABLE})

block_df = pd.DataFrame(rows)

summary = pd.concat([
    block_df.groupby('subject').size().rename('total_blocks'),
    block_df.groupby('subject')['dropped'].sum().rename('nan_blocks'),
], axis=1)
summary['flag_exclude'] = summary['nan_blocks'] >= EXCLUDE_THRESHOLD
summary = summary.sort_values('nan_blocks', ascending=False)

print(summary.to_string())
print()
print(f"Subjects flagged for exclusion (>= {EXCLUDE_THRESHOLD} dropped blocks):")
print(summary[summary['flag_exclude']].index.tolist())

# For Lauren: export new motion information as a CSV

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from parcellation import get_block_indices, extract_block_features, get_bold_paths
from events_loader import load_events_tsv, block_event_rows

FMRIPREP_ROOT = Path("/ix1/lhallion/TReW-Brain/preprocessed/")   # fmriprep root (contains sub-*/ses-01/func)
TASK_SPIKE_MASK_DIR = Path("/ihome/cladouceur/westbroo/Scripts/lhallion_scripts/TReW-Brain/dyscontrol_mvpa_featlike_nosmoothing/derivatives/parcels/yeo_2011_motiononly/")   # same dir process_run() wrote spike_mask.npy into

TR = 2.0
HRF_LAG_TRS = 3
MIN_FRACTION_USABLE = 0.7
REST_TASK = 'restingstate'
TASK_TASK = 'PDT'

def build_corrected_spike_mask(confounds_path):
    df = pd.read_csv(confounds_path, sep='\t')
    outlier_cols = [c for c in df.columns if c.startswith('motion_outlier')]
    if outlier_cols:
        return (df[outlier_cols].sum(axis=1) > 0).values
    return np.zeros(len(df), dtype=bool)

# Subject list, derived from existing task parcel outputs
task_files = sorted(TASK_SPIKE_MASK_DIR.glob('sub-*_run-*_spike_mask.npy'))
subjects = sorted({int(p.stem.split('sub-')[1].split('_')[0]) for p in task_files})

# --- Rest spikes ---
rest_rows = []
for sub in subjects:
    paths = get_bold_paths(FMRIPREP_ROOT, sub, run='01', task=REST_TASK)
    if not paths['confounds'].exists():
        rest_rows.append({'subject': sub, 'rest_n_trs': np.nan,
                           'rest_n_spike': np.nan, 'rest_pct_spike': np.nan})
        continue
    mask = build_corrected_spike_mask(paths['confounds'])
    rest_rows.append({'subject': sub, 'rest_n_trs': len(mask),
                       'rest_n_spike': int(mask.sum()),
                       'rest_pct_spike': round(100 * mask.mean(), 2)})
rest_df = pd.DataFrame(rest_rows)

# --- Task spikes (TR-level, summed across runs) ---
task_tr_rows = []
for spike_path in task_files:
    stem = spike_path.stem
    sub = int(stem.split('sub-')[1].split('_')[0])
    run = stem.split('run-')[1].split('_')[0]
    confounds_path = get_bold_paths(FMRIPREP_ROOT, sub, run, task=TASK_TASK)['confounds']
    if not confounds_path.exists():
        continue
    mask = build_corrected_spike_mask(confounds_path)
    task_tr_rows.append({'subject': sub, 'n_trs': len(mask), 'n_spike': int(mask.sum())})

task_spike_summary = (pd.DataFrame(task_tr_rows).groupby('subject')
                       .agg(task_n_trs=('n_trs', 'sum'), task_n_spike=('n_spike', 'sum'))
                       .assign(task_pct_spike=lambda d: round(100 * d['task_n_spike'] / d['task_n_trs'], 2))
                       .reset_index())

# --- Task block-level exclusion (Thoughtblock) ---
block_rows = []
for spike_path in task_files:
    stem = spike_path.stem
    sub = int(stem.split('sub-')[1].split('_')[0])
    run = stem.split('run-')[1].split('_')[0]
    confounds_path = get_bold_paths(FMRIPREP_ROOT, sub, run, task=TASK_TASK)['confounds']
    if not confounds_path.exists():
        continue
    mask = build_corrected_spike_mask(confounds_path)
    n_trs = len(mask)
    events_df = load_events_tsv(sub, run)
    tb_blocks = [b for b in block_event_rows(events_df) if b['condition'] == 'Thoughtblock']
    for block in tb_blocks:
        _, qc = extract_block_features(
            parcel_ts=np.zeros((n_trs, 1)),
            spike_mask=mask,
            onset_sec=block['onset'], duration_sec=block['duration'],
            tr=TR, hrf_lag_trs=HRF_LAG_TRS, min_fraction_usable=MIN_FRACTION_USABLE,
        )
        block_rows.append({'subject': sub, 'dropped': qc['fraction_usable'] < MIN_FRACTION_USABLE})

block_summary = (pd.DataFrame(block_rows).groupby('subject')
                  .agg(task_n_blocks_total=('dropped', 'size'),
                       task_n_blocks_excluded=('dropped', 'sum'))
                  .reset_index())

# --- Merge into one QC row per subject ---
qc_df = (pd.DataFrame({'subject': subjects})
          .merge(rest_df, on='subject', how='left')
          .merge(task_spike_summary, on='subject', how='left')
          .merge(block_summary, on='subject', how='left')
          .sort_values('subject'))

qc_df.to_csv('figure_data/subject_qc_summary.csv', index=False)
print(qc_df.to_string(index=False))
print("\nSaved to figure_data/subject_qc_summary.csv")